In [ ]:
# ─────────────────────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. LOAD & PREP
# ─────────────────────────────────────────────────────────────
df = pd.read_csv("../src/data/parkinsons_updrs.csv")

TARGET     = "total_UPDRS"
VOICE_COLS = ["Jitter(%)", "Jitter(Abs)", "Jitter:RAP", "Jitter:PPQ5", "Jitter:DDP",
              "Shimmer", "Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5",
              "Shimmer:APQ11", "Shimmer:DDA", "NHR", "HNR", "RPDE", "DFA", "PPE"]
META_COLS  = ["age", "sex"]   # kept outside PCA intentionally

# Age group for evaluation only (not fed to model as bins)
df["age_group"] = pd.cut(df["age"], bins=[0,60,70,120],
                          labels=["<60","60-70",">70"])

In [ ]:
# Sort by subject and time, then compute change from each subject's first recording
df = df.sort_values(["subject#", "test_time"])
df["baseline_UPDRS"] = df.groupby("subject#")["total_UPDRS"].transform("first")
df["delta_UPDRS"]    = df["total_UPDRS"] - df["baseline_UPDRS"]

# Now use delta_UPDRS as target — variance is much smaller and subject-agnostic
TARGET = "delta_UPDRS"

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2. SUBJECT-AWARE TRAIN / TEST SPLIT
#    42 subjects → 34 train, 8 test (80/20 at subject level)
# ─────────────────────────────────────────────────────────────
subjects     = df["subject#"].unique()
np.random.shuffle(subjects)
n_test       = max(1, int(len(subjects) * 0.2))   # 8 subjects
test_subjects  = subjects[:n_test]
train_subjects = subjects[n_test:]

train_df = df[df["subject#"].isin(train_subjects)].copy()
test_df  = df[df["subject#"].isin(test_subjects)].copy()

print(f"Train: {len(train_subjects)} subjects, {len(train_df)} rows")
print(f"Test : {len(test_subjects)}  subjects, {len(test_df)}  rows")
print(f"Test subjects: {sorted(test_subjects)}")

# Sanity check — no subject overlap
assert len(set(train_subjects) & set(test_subjects)) == 0, "Leakage!"

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3. PREPROCESSING
#    Voice features → StandardScaler → PCA (8 components)
#    age, sex       → StandardScaler  (kept separate)
#    Final input    → 8 PCs + age + sex = 10 features
# ─────────────────────────────────────────────────────────────
scaler_voice = StandardScaler()
scaler_meta  = StandardScaler()

# Fit ONLY on train
X_voice_train = scaler_voice.fit_transform(train_df[VOICE_COLS])
X_voice_test  = scaler_voice.transform(test_df[VOICE_COLS])

X_meta_train  = scaler_meta.fit_transform(train_df[META_COLS])
X_meta_test   = scaler_meta.transform(test_df[META_COLS])

# PCA on voice — fit on train only
pca = PCA(n_components=8, random_state=42)
X_pca_train = pca.fit_transform(X_voice_train)
X_pca_test  = pca.transform(X_voice_test)

print(f"\nPCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")
print(f"Per component: {np.round(pca.explained_variance_ratio_, 3)}")

# Concatenate: [8 PCs | age | sex] = 10 features
X_train = np.hstack([X_pca_train, X_meta_train]).astype(np.float32)
X_test  = np.hstack([X_pca_test,  X_meta_test ]).astype(np.float32)

y_train = train_df[TARGET].values.astype(np.float32)
y_test  = test_df[TARGET].values.astype(np.float32)

print(f"\nFinal input shape — train: {X_train.shape}, test: {X_test.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4. PYTORCH DATASET & DATALOADER
# ─────────────────────────────────────────────────────────────
def make_loader(X, y, batch_size=128, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X),
                       torch.from_numpy(y).unsqueeze(1))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5. MLP ARCHITECTURE
#    Input(10) → 64 → 128 → 64 → 32 → 1
#    BatchNorm + Dropout after each hidden layer
#    Small & regularised — appropriate for 5875 rows / 10 features
# ─────────────────────────────────────────────────────────────
class UPDRS_MLP(nn.Module):
    def __init__(self, input_dim=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            # Block 1
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Block 2
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Block 3
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Block 4
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            # Output
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

model_mlp = UPDRS_MLP(input_dim=X_train.shape[1]).to(DEVICE)
print(model_mlp)
total_params = sum(p.numel() for p in model_mlp.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6. TRAINING
# ─────────────────────────────────────────────────────────────
EPOCHS    = 200
LR        = 1e-3
WD        = 1e-4   # L2 regularisation

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_mlp.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=10, factor=0.5)

history = {"train_loss": [], "val_loss": [], "val_mae": [], "val_r2": []}

best_val_loss = float("inf")
best_state    = None
patience_ctr  = 0
EARLY_STOP    = 25

for epoch in range(1, EPOCHS + 1):
    # ── Train
    model_mlp.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model_mlp(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_loader.dataset)

    # ── Validate
    model_mlp.eval()
    val_preds, val_trues = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)
            val_preds.append(model_mlp(xb).cpu().numpy())
            val_trues.append(yb.numpy())

    val_preds = np.concatenate(val_preds).flatten()
    val_trues = np.concatenate(val_trues).flatten()
    val_loss  = mean_squared_error(val_trues, val_preds)
    val_mae   = mean_absolute_error(val_trues, val_preds)
    val_r2    = r2_score(val_trues, val_preds)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)
    history["val_r2"].append(val_r2)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.clone() for k, v in model_mlp.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= EARLY_STOP:
            print(f"Early stopping at epoch {epoch}")
            break

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Train MSE: {train_loss:.4f} | "
              f"Val MSE: {val_loss:.4f} | MAE: {val_mae:.4f} | R²: {val_r2:.4f}")

# Restore best weights
model_mlp.load_state_dict(best_state)
print(f"\nBest Val MSE: {best_val_loss:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 7. FINAL EVALUATION — OVERALL
# ─────────────────────────────────────────────────────────────
model_mlp.eval()
with torch.no_grad():
    y_pred = model_mlp(torch.from_numpy(X_test).to(DEVICE)).cpu().numpy().flatten()

overall_mae  = mean_absolute_error(y_test, y_pred)
overall_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
overall_r2   = r2_score(y_test, y_pred)

print("=== Overall MLP Performance ===")
print(f"  MAE : {overall_mae:.4f}")
print(f"  RMSE: {overall_rmse:.4f}")
print(f"  R²  : {overall_r2:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 8. PERFORMANCE ANALYSIS — BY AGE GROUP & GENDER
# ─────────────────────────────────────────────────────────────
meta_test = test_df[["subject#", "age", "sex", "age_group"]].copy().reset_index(drop=True)
meta_test["gender"]    = meta_test["sex"].map({0: "Male", 1: "Female"})
meta_test["y_true"]    = y_test
meta_test["y_pred"]    = y_pred
meta_test["residual"]  = y_test - y_pred
meta_test["abs_error"] = np.abs(meta_test["residual"])

def group_metrics(df_sub, label):
    rows = []
    for grp, g in df_sub.groupby(label, observed=True):
        if len(g) < 2:
            continue
        rows.append({
            label:   grp,
            "n":     len(g),
            "MAE":   mean_absolute_error(g["y_true"], g["y_pred"]),
            "RMSE":  np.sqrt(mean_squared_error(g["y_true"], g["y_pred"])),
            "R2":    r2_score(g["y_true"], g["y_pred"])
        })
    return pd.DataFrame(rows)

age_metrics    = group_metrics(meta_test, "age_group")
gender_metrics = group_metrics(meta_test, "gender")

print("\n=== Metrics by Age Group ===")
print(age_metrics.to_string(index=False))
print("\n=== Metrics by Gender ===")
print(gender_metrics.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────
# 9. PLOTS
# ─────────────────────────────────────────────────────────────
age_palette    = {"<60": "#EF5350", "60-70": "#42A5F5", ">70": "#26A69A"}
gender_palette = {"Male": "#42A5F5", "Female": "#EF5350"}
epochs_range   = range(1, len(history["train_loss"]) + 1)

# 9a. Learning curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_range, history["train_loss"], label="Train MSE", color="#42A5F5")
axes[0].plot(epochs_range, history["val_loss"],   label="Val MSE",   color="#EF5350")
axes[0].set_title("Loss (MSE)"); axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs_range, history["val_mae"], color="#FF7043")
axes[1].set_title("Val MAE per Epoch"); axes[1].set_xlabel("Epoch")

axes[2].plot(epochs_range, history["val_r2"], color="#26A69A")
axes[2].set_title("Val R² per Epoch"); axes[2].set_xlabel("Epoch")

plt.suptitle("MLP Training History", fontsize=13)
plt.tight_layout(); plt.savefig("mlp_learning_curves.png", dpi=150); plt.show()

# 9b. Predicted vs Actual — coloured by age group
plt.figure(figsize=(7, 6))
for grp, g in meta_test.groupby("age_group", observed=True):
    plt.scatter(g["y_true"], g["y_pred"],
                alpha=0.4, s=18, label=grp, color=age_palette[str(grp)])
lims = [meta_test[["y_true","y_pred"]].min().min(),
        meta_test[["y_true","y_pred"]].max().max()]
plt.plot(lims, lims, "k--", linewidth=1.5, label="Perfect")
plt.xlabel("Actual total_UPDRS"); plt.ylabel("Predicted total_UPDRS")
plt.title("MLP — Predicted vs Actual by Age Group")
plt.legend(); plt.tight_layout()
plt.savefig("mlp_pred_vs_actual.png", dpi=150); plt.show()

# 9c. Residuals by age group
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, grp in zip(axes, ["<60", "60-70", ">70"]):
    g = meta_test[meta_test["age_group"] == grp]
    ax.scatter(g["y_pred"], g["residual"],
               alpha=0.4, s=15, color=age_palette[grp])
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Residuals — Age {grp}  (n={len(g)})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Residual")
plt.suptitle("MLP Residuals by Age Group", fontsize=13)
plt.tight_layout(); plt.savefig("mlp_residuals_age.png", dpi=150); plt.show()

# 9d. Error heatmap — age group × gender
import seaborn as sns
pivot = meta_test.groupby(["age_group", "gender"],
                           observed=True)["abs_error"].mean().unstack()
plt.figure(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.5, cbar_kws={"label": "Mean Abs Error"})
plt.title("MLP — Mean Absolute Error: Age Group × Gender")
plt.tight_layout(); plt.savefig("mlp_error_heatmap.png", dpi=150); plt.show()

# 9e. Error violin — by age group
plt.figure(figsize=(9, 5))
sns.violinplot(data=meta_test, x="age_group", y="abs_error",
               order=["<60","60-70",">70"],
               palette=age_palette, inner="box", cut=0)
plt.title("MLP — Absolute Error Distribution by Age Group")
plt.xlabel("Age Group"); plt.ylabel("Absolute Error")
plt.tight_layout(); plt.savefig("mlp_error_violin.png", dpi=150); plt.show()